ake time
# 04 - Destroy Infrastructure

Use this notebook to clean up all resources created by the demo.

**Why this exists:** prototype demos should include a clear teardown path to avoid ongoing charges.

---

## Step 1 - Load environment and resolve Azure CLI

In [ ]:
import json
import os
import re
import shutil
import subprocess
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

def resolve_az_cli():
    az_in_path = shutil.which("az")
    if az_in_path:
        return az_in_path
    windows_fallbacks = [
        Path(r"C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
        Path(r"C:\\Program Files (x86)\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
    ]
    for candidate in windows_fallbacks:
        if candidate.exists():
            return str(candidate)
    return None

def extract_resource_group_from_deploy_notebook():
    deploy_nb = Path("../notebooks/01_deploy_infra.ipynb")
    if not deploy_nb.exists():
        return None
    data = json.loads(deploy_nb.read_text(encoding="utf-8"))
    for cell in data.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "\n".join(cell.get("source", []))
        match = re.search(r'^RESOURCE_GROUP\s*=\s*"([^"]+)"', source, re.MULTILINE)
        if match:
            return match.group(1)
    return None

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install Azure CLI and rerun.")

resource_group = os.getenv("AZURE_RESOURCE_GROUP") or extract_resource_group_from_deploy_notebook()
if not resource_group:
    raise RuntimeError(
        "Could not determine resource group. Set AZURE_RESOURCE_GROUP in ../env/.env "
        "or define RESOURCE_GROUP in 01_deploy_infra.ipynb."
    )

exists_check = subprocess.run(
    [az_cmd, "group", "exists", "--name", resource_group],
    capture_output=True,
    text=True,
    check=True,
 )
if exists_check.stdout.strip().lower() != "true":
    raise RuntimeError(f"Resource group '{resource_group}' does not exist in the current subscription.")

print(f"Using Azure CLI: {az_cmd}")
print(f"Resource group to delete: {resource_group}")

## Step 2 - Review resources before deletion

In [ ]:
subprocess.run([az_cmd, "resource", "list", "--resource-group", resource_group, "--output", "table"], check=False)

## Step 3 - Delete resources (keeping resource group for purge)

Set `CONFIRM_DELETE = True` to delete resources. The resource group will be kept for the purge operation and removed in Step 5.

In [ ]:
CONFIRM_DELETE = True

if not CONFIRM_DELETE:
    print("Deletion skipped. Set CONFIRM_DELETE = True to proceed.")
else:
    # List all resources in the resource group, then delete each one
    list_result = subprocess.run(
        [az_cmd, "resource", "list", "--resource-group", resource_group, "--output", "json"],
        capture_output=True,
        text=True,
        check=True,
    )
    resources = json.loads(list_result.stdout or "[]")
    
    if not resources:
        print(f"No resources found in {resource_group}.")
    else:
        # Delete each resource individually (keeps the resource group for purge operation)
        for resource in resources:
            resource_id = resource.get("id")
            resource_name = resource.get("name")
            if not resource_id:
                continue
            
            print(f"Deleting: {resource_name}...", end=" ")
            delete_result = subprocess.run(
                [az_cmd, "resource", "delete", "--ids", resource_id, "--yes"],
                capture_output=True,
                text=True,
                check=False,
            )
            if delete_result.returncode == 0:
                print("✓")
            else:
                print(f"⚠ (exit code {delete_result.returncode})")
        
        print(f"\nAll resources deleted from {resource_group}. Resource group kept for purge operation.")


## Step 4 - Purge soft-deleted AI resources (optional)

Set the confirmation flag in the next cell only if you want to permanently purge soft-deleted Cognitive Services accounts.

This is useful when you need the account name to be reusable immediately.

In [ ]:
import time
from urllib.parse import urlparse

CONFIRM_PURGE_SOFT_DELETES = True

ai_endpoint = os.getenv("AZURE_AI_ENDPOINT", "").strip()
target_account_name = ""
if ai_endpoint:
    target_account_name = urlparse(ai_endpoint).netloc.split(".")[0]

def list_deleted_accounts():
    deleted_result = subprocess.run(
        [az_cmd, "cognitiveservices", "account", "list-deleted", "--output", "json"],
        capture_output=True,
        text=True,
        check=True,
    )
    return json.loads(deleted_result.stdout or "[]")

deleted_accounts = list_deleted_accounts()

# Deletion can take time to surface in list-deleted; poll briefly for the specific account.
if target_account_name:
    for _ in range(12):
        candidates = [
            item for item in deleted_accounts
            if str(item.get("name", "")).lower() == target_account_name.lower()
        ]
        if candidates:
            break
        time.sleep(10)
        deleted_accounts = list_deleted_accounts()
else:
    candidates = deleted_accounts

if not candidates:
    if target_account_name:
        print(f"No soft-deleted Cognitive Services account found for: {target_account_name}")
    else:
        print("No soft-deleted Cognitive Services accounts found.")
elif not CONFIRM_PURGE_SOFT_DELETES:
    print("Purge skipped. Set CONFIRM_PURGE_SOFT_DELETES = True to permanently purge:")
    for item in candidates:
        print(f"- {item.get('name')} ({item.get('location')})")
else:
    for item in candidates:
        name = item.get("name")
        location = item.get("location")
        if not name or not location:
            print(f"Skipping invalid entry: {item}")
            continue

        purge_result = subprocess.run(
            [
                az_cmd,
                "cognitiveservices",
                "account",
                "purge",
                "--name",
                name,
                "--location",
                location,
                "--resource-group",
                resource_group,
            ],
            capture_output=True,
            text=True,
            check=False,
        )
        if purge_result.returncode != 0:
            raise RuntimeError(
                f"Failed to purge soft-deleted Cognitive Services account '{name}'.\n"
                f"stderr: {purge_result.stderr.strip()}\n"
                f"stdout: {purge_result.stdout.strip()}"
            )
        print(f"Purged soft-deleted Cognitive Services account: {name} ({location})")

print("\nStep 4 complete. Run Step 5 to delete the resource group.")

## Step 5 - Delete resource group

Run this cell after Step 4 completes to remove the resource group.

In [ ]:
CONFIRM_DELETE_RG = True

if not CONFIRM_DELETE_RG:
    print("Resource group deletion skipped. Set CONFIRM_DELETE_RG = True to proceed.")
else:
    delete_rg_result = subprocess.run(
        [az_cmd, "group", "delete", "--name", resource_group, "--yes"],
        capture_output=True,
        text=True,
        check=False,
    )
    if delete_rg_result.returncode == 0:
        print(f"Resource group '{resource_group}' deleted successfully.")
        print("Cleanup complete!")
    else:
        print(f"WARNING: Resource group deletion returned exit code {delete_rg_result.returncode}")
        print(f"stderr: {delete_rg_result.stderr}")

## Step 6 - Verify no soft-deleted resources remain

Confirm that all soft-deleted Cognitive Services accounts have been permanently purged.

In [ ]:
remaining_deleted = subprocess.run(
    [az_cmd, "cognitiveservices", "account", "list-deleted", "--output", "json"],
    capture_output=True,
    text=True,
    check=False,
)

if remaining_deleted.returncode == 0:
    remaining_accounts = json.loads(remaining_deleted.stdout or "[]")
    if remaining_accounts:
        print(f"WARNING: {len(remaining_accounts)} soft-deleted Cognitive Services account(s) remain:")
        for item in remaining_accounts:
            print(f"  - {item.get('name')} ({item.get('location')})")
    else:
        print("✓ Verification complete: No soft-deleted Cognitive Services accounts found.")
        print("✓ All demo resources have been successfully cleaned up!")
else:
    print(f"WARNING: Could not verify soft-deleted accounts (exit code {remaining_deleted.returncode})")
    print(f"stderr: {remaining_deleted.stderr}")